# Data Wrangling #

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
import re

# Connect to the database
conn = sqlite3.connect("./data/nutrition.db")
cur = conn.cursor()

In [7]:
food = pd.read_sql_query("SELECT * FROM food", conn)
food_nutrient = pd.read_sql_query("SELECT * FROM food_nutrient", conn)
nutrient = pd.read_sql_query("SELECT * FROM nutrient", conn)
wal_price = pd.read_sql_query("SELECT * FROM walmart_price", conn)
wf_price = pd.read_sql_query("SELECT * FROM wholefoods_price", conn)

In [ ]:
def normalize_text(str):
    if pd.isna(str):
        return ""
    str = str.lower()
    str = re.sub(r'[^a-z0-9\s]', ' ', str)
    str = re.sub(r'\s+', ' ', str).strip()
    return str

food["clean_desc"] = food["description"].apply(normalize_text)
wal_price["clean_name"] = wal_price["product_name"].apply(normalize_text)
wf_price["clean_name"] = wf_price["product_name"].apply(normalize_text)

food["clean_brand_owner"] = food["brand_owner"].apply(normalize_text)
food["clean_brand"] = food["brand_name"].apply(normalize_text)
food["clean_subbrand"] = food["subbrand_name"].apply(normalize_text)

wal_price["clean_brand"] = wal_price["brand"].apply(normalize_text)
wf_price["clean_brand"] = wf_price["brand"].apply(normalize_text)

In [ ]:
from rapidfuzz import process

cand_count = 0

food_names = food["clean_desc"].tolist()

def get_candidates(query, k=5):
    cand_count += 1

    matches = process.extract(query, food_names, limit=k)
    return [(m[0], m[1]) for m in matches]

wf_price["candidates"] = wf_price["clean_name"].apply(get_candidates)

KeyboardInterrupt: 

In [ ]:
def compute_score(store_row, food_row, text_score):
    text_sim = text_score / 100

    brand_sim = 1 if store_row["clean_brand"] == food_row["clean_brand_owner"] or store_row["clean_brand"] == food_row["clean_brand"] or store_row["clean_brand"] == food_row["clean_subbrand"] else 0

    return 0.7 * text_sim + 0.3 * brand_sim

In [ ]:
def match_product(row, score_thresh):
    best_score = score_thresh
    best_fdc = None

    for cand_name, text_score in row["candidates"]:
        food_row = food[food["clean_desc"] == cand_name].iloc[0]

        score = compute_score(row, food_row, text_score)

        if score > best_score:
            best_score = score
            best_fdc = food_row["fdc_id"]

    return best_fdc, best_score

wal_price[["fdc_id", "match_score"]] = wal_price.apply(
    lambda row: pd.Series(match_product(row, 0)),
    axis=1
)

In [ ]:
conn.close()